<a href="https://colab.research.google.com/github/team-telnyx/telnyx-code-examples/blob/main/cookbooks/inference/02_Structured_JSON_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured JSON Extraction with Telnyx Inference

In this notebook, you’ll use Telnyx Inference to extract structured JSON from messy, unstructured text.

This is useful for workflows like:

- Support ticket triage
- Email parsing
- Lead qualification
- Contract or document extraction
- Incident report summarization

We’ll use the OpenAI Python SDK with Telnyx’s OpenAI-compatible endpoint.


### Install SDK

In [ ]:
!pip install openai

In [ ]:
from getpass import getpass

TELNYX_API_KEY = getpass("Enter your Telnyx API key: ")


This notebook uses the OpenAI Python SDK because Telnyx Inference supports OpenAI-compatible APIs.


### Create Telnyx Client

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=TELNYX_API_KEY,
    base_url="https://api.telnyx.com/v2/ai/openai"
)

This example uses `moonshotai/Kimi-K2.6` for JSON extraction. You can swap in another Telnyx-hosted chat model, but output quality and formatting may vary by model.


### Define Messy Input Text

In [ ]:
ticket = """
Subject: URGENT - checkout failures after key rotation??

Hey support,

We rotated our Telnyx API key yesterday around 6:15pm ET as part of a security cleanup.
Since then, some of our backend jobs are failing when they try to send verification messages.

The weird part: our staging environment works fine, but production is throwing 401s.
We also saw a few 429s earlier this morning, but those might be from our retry loop going crazy.

Impact: new users can't complete signup because they never receive the verification code.
This started around 9:30am ET today and is affecting our US traffic only as far as we can tell.

Can someone check whether our old key is still cached somewhere, or if we missed a permission
on the new key? Happy to jump on a call. This is pretty urgent because paid acquisition is live today.

Account: Acme Health
Request ID examples: req_7K2... and req_9Q1...
"""


### Define the JSON Schema

In [ ]:
ticket_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "company": {"type": "string"},
        "category": {
            "type": "string",
            "enum": ["authentication", "rate_limit", "billing", "bug", "feature_request", "general"]
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high", "urgent"]
        },
        "summary": {"type": "string"},
        "affected_environment": {"type": "string"},
        "affected_region": {"type": "string"},
        "customer_impact": {"type": "string"},
        "error_codes": {
            "type": "array",
            "items": {"type": "string"}
        },
        "suspected_cause": {"type": "string"},
        "requested_action": {"type": "string"}
    },
    "required": [
        "company",
        "category",
        "priority",
        "summary",
        "affected_environment",
        "affected_region",
        "customer_impact",
        "error_codes",
        "suspected_cause",
        "requested_action"
    ]
}

### Extract Structured JSON

In [ ]:
response = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6", # Telnyx-hosted chat model used in this example
    messages=[
        {
            "role": "system",
            "content": """
You extract structured information from support tickets.

Return only valid JSON.
Do not use Markdown.
Do not include extra fields.

Use exactly this JSON shape:

{
  "company": "string",
  "category": "authentication | rate_limit | billing | bug | feature_request | general",
  "priority": "low | medium | high | urgent",
  "summary": "string",
  "affected_environment": "string",
  "affected_region": "string",
  "customer_impact": "string",
  "error_codes": ["string"],
  "suspected_cause": "string",
  "requested_action": "string"
}
"""
        },
        {
            "role": "user",
            "content": ticket
        }
    ],
    temperature=0.0,
    response_format={"type": "json_object"},
)

raw_output = response.choices[0].message.content
print(raw_output)


### Parse the JSON


In [ ]:
import json
from pprint import pprint

structured_ticket = json.loads(raw_output)

pprint(structured_ticket)

## Example Output

You should see a JSON object with the same fields as the requested shape:

```json
{
  "company": "Acme Health",
  "category": "authentication",
  "priority": "urgent",
  "summary": "Production webhook jobs are failing after an API key rotation.",
  "affected_environment": "production",
  "affected_region": "US",
  "customer_impact": "New users cannot complete signup because verification codes are not being sent.",
  "error_codes": ["401", "429"],
  "suspected_cause": "The new API key may be missing permissions, or the old key may still be cached somewhere.",
  "requested_action": "Check authentication configuration and help debug the API key rotation issue."
}
```

Your exact wording may vary, but the field names should match this structure.

## Try Your Own Text

Replace the text below with another support ticket, email, or customer message.


In [ ]:
import re
custom_text = """
Subject: Billing question - duplicate charge

Hi, I noticed two charges from Telnyx on my card this month.
One was on May 2 and another was on May 4 for the same amount.

Our finance team is asking whether this is expected or if one charge was accidental.
Nothing is broken right now, but we'd like to resolve it before our books close this week.

Company: Northstar Logistics
"""

response = client.chat.completions.create(
    model="moonshotai/Kimi-K2.6",
    messages=[
        {
            "role": "system",
            "content": (
                "Extract structured information from the user's message. "
                "Return only a valid JSON object. Do not wrap it in Markdown. "
                "Do not include explanations, comments, or code fences."
            )
        },
        {
            "role": "user",
            "content": custom_text
        }
    ],
    temperature=0.0,
    extra_body={
        "guided_json": ticket_schema
    }
)

raw_output = response.choices[0].message.content.strip()
raw_output = re.sub(r"^```(?:json)?\s*", "", raw_output)
raw_output = re.sub(r"\s*```$", "", raw_output)

structured_result = json.loads(raw_output)

pprint(structured_result)

### Troubleshooting

In [ ]:
print(repr(response.choices[0].message.content))


Common causes:

- The response was wrapped in Markdown code fences.
- The response included extra text before or after the JSON.
- The schema was changed in a way that makes generation harder.

The parsing cells above remove basic Markdown code fences before calling `json.loads()`.


## Explore More

Want to build more with Telnyx Inference?

- Browse Telnyx Inference docs: https://developers.telnyx.com/docs/inference
- Explore Telnyx examples on GitHub: https://github.com/team-telnyx/telnyx-code-examples
- Try the Telnyx Mission Control Portal: https://portal.telnyx.com/